In [1]:
import json
import numpy as np

# Load previously filtered, valid poses
with open(r"E:\Sanjay\Dodeca\valid_dodecahedron_poses_SQPNP.json", "r") as f:
    valid_poses = json.load(f)

# Extract rotation and translation for each pose
R_list = []
t_list = []

for pose in valid_poses:
    R_list.append(np.array(pose["R"], dtype=np.float64))
    t_list.append(np.array(pose["t"], dtype=np.float64).reshape(3,1))

R_list = np.array(R_list)   # shape: (m, 3, 3)
t_list = np.array(t_list)   # shape: (m, 3, 1)

print(f"Loaded {len(R_list)} valid poses for pen-tip calibration")

Loaded 72 valid poses for pen-tip calibration


In [2]:
import itertools

m = len(R_list)
A_rows = []
b_rows = []

# Loop over all unique pairs (i < j)
for i, j in itertools.combinations(range(m), 2):
    Ri, Rj = R_list[i], R_list[j]
    ti, tj = t_list[i], t_list[j]

    # Linear equation: (Ri - Rj) c = tj - ti
    A_rows.append(Ri - Rj)
    b_rows.append(tj - ti)

# Stack into big matrices
A = np.vstack(A_rows)   # shape: (3*N, 3)
b = np.vstack(b_rows)   # shape: (3*N, 1)

# Solve least-squares
c, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)

print("Estimated pen-tip position in dodecahedron frame:")
print(c.flatten())

Estimated pen-tip position in dodecahedron frame:
[ 5.19330354e-02 -1.69081472e+00  1.85539596e+02]


In [3]:
from scipy.optimize import least_squares

def residual_func(c_flat, A, b):
    c = c_flat.reshape(3,1)
    return (A @ c - b).flatten()

# Initial guess from your current result
c0 = np.array([17.059, 3.619, -167.552])

res = least_squares(
    residual_func,
    c0,
    args=(A, b),
    loss='huber',          # robust to outliers
    f_scale=1.0,           # tune based on expected residual scale (~few mm)
    verbose=1
)

c_robust = res.x
print("Robust estimate:", c_robust)
print("Robust cost:", res.cost)

`ftol` termination condition is satisfied.
Function evaluations 5, initial cost 3.1615e+03, final cost 3.1429e+03, first-order optimality 7.22e-07.
Robust estimate: [  16.45458937    3.69424469 -168.27762904]
Robust cost: 3142.8817274808403


In [3]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import glob
calib = np.load(r"E:\Sanjay\Dodeca\realsense_color_intrinsics_1280x720.npz")
K = calib["camera_matrix"]
D = calib["dist_coeffs"]
image_paths = sorted(glob.glob("dodeca_pentip/*.jpg"))
# Assuming c, R_list, t_list, K, D are defined
for idx, img_path in enumerate(image_paths):
    img = cv2.imread(img_path)
    if img is None:
        continue

    Rk, tk = R_list[idx], t_list[idx]
    c_world = Rk @ c + tk   # 3D world position of pen tip

    # Project to 2D
    proj, _ = cv2.projectPoints(c_world.reshape(1,3), np.zeros(3), np.zeros(3), K, D)
    x, y = proj.ravel().astype(int)

    # Draw on image
    img_vis = img.copy()
    cv2.circle(img_vis, (x, y), 5, (0,0,255), -1)
    plt.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
    plt.title(f"Frame {idx} — Pen Tip Projection")
    plt.show()

In [5]:
z_values = [(Rk @ c + tk)[2] for Rk, tk in zip(R_list, t_list)]
print("Z-values of pen tip in world frame:", z_values)
print("Std deviation (should be small):", np.std(z_values))
print("Rank of A:", np.linalg.matrix_rank(A))  # Should be 3
cond_number = np.linalg.cond(A)
print("Condition number of A:", cond_number)
residual_norm = np.linalg.norm(A @ c - b)
print("Residual norm:", residual_norm)
pen_length = np.linalg.norm(c)
print("Distance from dodecahedron center to pen tip:", pen_length, "mm")

Z-values of pen tip in world frame: [array([485.89693694]), array([492.40394851]), array([487.30245224]), array([489.39397409]), array([489.77452371]), array([489.91558872]), array([488.01892268]), array([490.01214466]), array([490.90809986]), array([489.38984522]), array([490.96131501]), array([489.407104]), array([490.19533152]), array([492.1010419]), array([491.75091859]), array([487.62832467]), array([488.78862612]), array([491.8635935]), array([488.37517054]), array([487.9769297]), array([487.24385444]), array([489.04008543]), array([488.03365891]), array([487.0882216]), array([486.83128612]), array([488.24925943]), array([488.63409309])]
Std deviation (should be small): 1.7006087324160146
Rank of A: 3
Condition number of A: 2.0107180741303634
Residual norm: 67.70937382920297
Distance from dodecahedron center to pen tip: 184.99774357961917 mm


In [1]:
import os
import shutil
import re

# ──────────────────────────────────────────────
#  CHANGE THESE 3 LINES ONLY
# ──────────────────────────────────────────────

MAIN_FOLDER     = r"E:\Sanjay\Dodeca\dodeca_dataset"          # ← your source folder
NEW_GOOD_FOLDER = r"E:\Sanjay\Dodeca\dodeca_validate_pentip"    # ← where to save selected images
ACTION          = "copy"                                 # "copy" or "move"

# Create destination folder if it doesn't exist
os.makedirs(NEW_GOOD_FOLDER, exist_ok=True)

# ──────────────────────────────────────────────
# Your log lines (only the [OK] ones matter)
# ──────────────────────────────────────────────

log = """
[OK] dodeca_0.jpg | RMS = 0.50px
[OK] dodeca_1.jpg | RMS = 0.74px
[OK] dodeca_10.jpg | RMS = 0.49px
[OK] dodeca_11.jpg | RMS = 0.49px
[OK] dodeca_12.jpg | RMS = 0.46px
[OK] dodeca_13.jpg | RMS = 0.60px
[OK] dodeca_15.jpg | RMS = 0.33px
[OK] dodeca_16.jpg | RMS = 0.44px
[OK] dodeca_17.jpg | RMS = 0.55px
[OK] dodeca_18.jpg | RMS = 0.51px
[OK] dodeca_19.jpg | RMS = 0.45px
[OK] dodeca_2.jpg | RMS = 0.41px
[OK] dodeca_20.jpg | RMS = 0.61px
[OK] dodeca_21.jpg | RMS = 0.58px
[OK] dodeca_22.jpg | RMS = 0.51px
[OK] dodeca_23.jpg | RMS = 0.43px
[OK] dodeca_24.jpg | RMS = 0.42px
[OK] dodeca_25.jpg | RMS = 0.49px
[OK] dodeca_26.jpg | RMS = 0.45px
[OK] dodeca_27.jpg | RMS = 0.54px
[OK] dodeca_28.jpg | RMS = 0.56px
[OK] dodeca_29.jpg | RMS = 0.52px
[OK] dodeca_3.jpg | RMS = 0.50px
[OK] dodeca_30.jpg | RMS = 0.52px
[OK] dodeca_31.jpg | RMS = 0.29px
[OK] dodeca_32.jpg | RMS = 0.48px
[OK] dodeca_33.jpg | RMS = 0.32px
"""

# ──────────────────────────────────────────────
# Extract only filenames that appear as [OK]
# ──────────────────────────────────────────────

good_files = set()
pattern = r"dodeca_\d+\.jpg"

for line in log.splitlines():
    line = line.strip()
    if line.startswith("[OK]"):
        match = re.search(pattern, line)
        if match:
            good_files.add(match.group(0))

print(f"Found {len(good_files)} images marked [OK] in log\n")

# ──────────────────────────────────────────────
# Copy / Move only the matching files
# ──────────────────────────────────────────────

copied_count = 0
not_found_count = 0

for filename in sorted(good_files):
    src_path = os.path.join(MAIN_FOLDER, filename)
    dst_path = os.path.join(NEW_GOOD_FOLDER, filename)

    if os.path.isfile(src_path):
        if ACTION == "move":
            shutil.move(src_path, dst_path)
            action_word = "Moved"
        else:
            shutil.copy2(src_path, dst_path)
            action_word = "Copied"

        print(f"{action_word}: {filename}")
        copied_count += 1
    else:
        print(f"Not found : {filename}")
        not_found_count += 1

print("\n" + "─" * 50)
print(f"Finished: {copied_count} files {ACTION}ed successfully")
print(f"         {not_found_count} files not found in source folder")
print("─" * 50)

Found 27 images marked [OK] in log

Copied: dodeca_0.jpg
Copied: dodeca_1.jpg
Copied: dodeca_10.jpg
Copied: dodeca_11.jpg
Copied: dodeca_12.jpg
Copied: dodeca_13.jpg
Copied: dodeca_15.jpg
Copied: dodeca_16.jpg
Copied: dodeca_17.jpg
Copied: dodeca_18.jpg
Copied: dodeca_19.jpg
Copied: dodeca_2.jpg
Copied: dodeca_20.jpg
Copied: dodeca_21.jpg
Copied: dodeca_22.jpg
Copied: dodeca_23.jpg
Copied: dodeca_24.jpg
Copied: dodeca_25.jpg
Copied: dodeca_26.jpg
Copied: dodeca_27.jpg
Copied: dodeca_28.jpg
Copied: dodeca_29.jpg
Copied: dodeca_3.jpg
Copied: dodeca_30.jpg
Copied: dodeca_31.jpg
Copied: dodeca_32.jpg
Copied: dodeca_33.jpg

──────────────────────────────────────────────────
Finished: 27 files copyed successfully
         0 files not found in source folder
──────────────────────────────────────────────────


In [3]:
import itertools
from scipy.optimize import least_squares, minimize
# ────────────────────────────────────────────────
# 1. Load data
# ────────────────────────────────────────────────
json_path = r"E:\Sanjay\Dodeca\valid_dodecahedron_poses.json"

with open(json_path, "r") as f:
    valid_poses = json.load(f)

R_list = []
t_list = []
rms_list = []

for pose in valid_poses:
    R_list.append(np.array(pose["R"], dtype=np.float64))
    t_list.append(np.array(pose["t"], dtype=np.float64).reshape(3, 1))
    rms_list.append(pose.get("rms", -1))  # optional

R_list = np.array(R_list)          # (N, 3, 3)
t_list = np.array(t_list)          # (N, 3, 1)
rms_list = np.array(rms_list)

N = len(R_list)
print(f"Loaded {N} valid poses (RMS range: {rms_list.min():.3f} – {rms_list.max():.3f} px)")
# ────────────────────────────────────────────────
# 2. Build pairwise equations with angular filtering
# ────────────────────────────────────────────────
MIN_ANGLE_DEG = 12.0   # ← tune: 10–20° is usually good

A_rows = []
b_rows = []
used_pairs = 0

for i, j in itertools.combinations(range(N), 2):
    Ri, Rj = R_list[i], R_list[j]
    ti, tj = t_list[i], t_list[j]

    # Rotation angle between Ri and Rj
    trace = np.clip((np.trace(Ri.T @ Rj) - 1.0) / 2.0, -1.0, 1.0)
    angle_deg = np.degrees(np.arccos(trace))

    if angle_deg >= MIN_ANGLE_DEG:
        A_rows.append(Ri - Rj)
        b_rows.append(tj - ti)
        used_pairs += 1

if used_pairs == 0:
    raise ValueError("No pose pairs met the minimum angle threshold. Lower MIN_ANGLE_DEG or get more diverse poses.")

A = np.vstack(A_rows)     # (3*used_pairs, 3)
b = np.vstack(b_rows)     # (3*used_pairs, 1)

print(f"Used {used_pairs} / {N*(N-1)//2} pairs (min angle = {MIN_ANGLE_DEG}°)")
print(f"Condition number of A: {np.linalg.cond(A):.2f}")
print(f"Rank of A: {np.linalg.matrix_rank(A)} (should be 3)")

Loaded 23 valid poses (RMS range: 1.157 – 2.740 px)
Used 241 / 253 pairs (min angle = 12.0°)
Condition number of A: 3.33
Rank of A: 3 (should be 3)


In [4]:
# ────────────────────────────────────────────────
# 3. Robust linear solve (Huber loss)
# ────────────────────────────────────────────────
def linear_residual(c_flat):
    c = c_flat.reshape(3, 1)
    return (A @ c - b).ravel()

c0 = np.array([0.0, 0.0, -150.0])   # rough guess (negative Z)

res_robust = least_squares(
    linear_residual,
    c0,
    loss='huber',
    f_scale=5.0,                    # tune: expected residual scale in mm (~3–10)
    verbose=1,
    ftol=1e-10,
    xtol=1e-10
)

c_robust = res_robust.x.reshape(3, 1)
print("\nRobust linear estimate:")
print(c_robust.flatten())
print(f"Robust cost: {res_robust.cost:.4f}")
print(f"Pen length: {np.linalg.norm(c_robust):.3f} mm")

`xtol` termination condition is satisfied.
Function evaluations 11, initial cost 3.7770e+04, final cost 1.0678e+04, first-order optimality 2.47e-06.

Robust linear estimate:
[  16.47401288    3.80369603 -167.76269026]
Robust cost: 10677.8747
Pen length: 168.613 mm


In [5]:
# ────────────────────────────────────────────────
# 4. Nonlinear refinement: minimize variance of tip position
# ────────────────────────────────────────────────
def variance_loss(c_flat):
    c = c_flat.reshape(3, 1)
    tip_positions = []
    for R, t in zip(R_list, t_list):
        tip_cam = R @ c + t           # 3×1
        tip_positions.append(tip_cam.ravel())
    tip_positions = np.array(tip_positions)  # (N, 3)
    mean_tip = np.mean(tip_positions, axis=0)
    variances = np.mean((tip_positions - mean_tip)**2, axis=0)
    return np.sum(variances)  # or np.linalg.norm(variances)

res_nonlin = minimize(
    variance_loss,
    c_robust.ravel(),
    method='Nelder-Mead',
    options={'maxiter': 5000, 'disp': True, 'fatol': 1e-6}
)

c_final = res_nonlin.x.reshape(3, 1)
print("\nFinal nonlinear refined estimate:")
print(c_final.flatten())
print(f"Optimization success: {res_nonlin.success}")
print(f"Final variance loss: {res_nonlin.fun:.6f}")

Optimization terminated successfully.
         Current function value: 72.124284
         Iterations: 65
         Function evaluations: 119

Final nonlinear refined estimate:
[  17.05950302    3.61899701 -167.5519474 ]
Optimization success: True
Final variance loss: 72.124284


In [6]:
# ────────────────────────────────────────────────
# 5. Evaluate quality
# ────────────────────────────────────────────────
tip_positions = []
for R, t in zip(R_list, t_list):
    tip_cam = R @ c_final + t
    tip_positions.append(tip_cam.ravel())

tip_positions = np.array(tip_positions)  # (N, 3)

mean_tip = np.mean(tip_positions, axis=0)
std_tip  = np.std(tip_positions, axis=0)
range_tip = np.max(tip_positions, axis=0) - np.min(tip_positions, axis=0)

print("\nReconstructed tip position statistics (camera frame):")
print(f"Mean XYZ : {mean_tip}")
print(f"Std  XYZ : {std_tip}   → overall std = {np.linalg.norm(std_tip):.3f} mm")
print(f"Range XYZ: {range_tip}")
print(f"Pen length (‖c‖) = {np.linalg.norm(c_final):.3f} mm")


Reconstructed tip position statistics (camera frame):
Mean XYZ : [  37.19249518  -74.13484673 -413.43582095]
Std  XYZ : [4.72311685 2.79922685 6.47925771]   → overall std = 8.493 mm
Range XYZ: [24.47512466 14.85046714 37.04759952]
Pen length (‖c‖) = 168.457 mm
